In [1]:
import requests
import os
from dotenv import load_dotenv
import json
import pandas as pd
import datetime
import time

EXTRACT

In [2]:
#loading API key from secrets/.env
load_dotenv()
API_KEY =  os.getenv("API_KEY")

In [ ]:
base_url = "https://backend.simfin.com/api/v3/companies"

endpoint_url= f"{base_url}/prices/compact"

headers = {"accept": "application/json",
            "Authorization": f"{API_KEY}"}

In [ ]:
#Creating PySimFin class. This is the API wrapper we will use to make API calls.

class PySimFin:
    def __init__(self):
        self.endpoint_url = f"{endpoint_url}"
        self.headers = headers.copy()   

    def get_share_prices(self, ticker: str):
        params = {'ticker': ticker}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

    def get_share_prices_verbose(self, ticker: str, start: str, end: str):
        params = {'ticker': ticker,
                  'start': start,
                  'end': end}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
    

    def get_share_prices_today(self, ticker: str):
        today = str(datetime.datetime.today()).split()[0] #gets today's date
        params = {'ticker': ticker,
                  'start': today}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

In [ ]:
# This will enventually need to go to the config.yaml
top_40_companies =[
{"name" : "Nvidia","ticker": "NVDA"},
{"name" : "Apple Inc.","ticker": "AAPL"},
{"name" : "Microsoft",	"ticker": "MSFT"},
{"name" : "Amazon",	"ticker": "AMZN"},
{"name" : "Meta Platforms",	"ticker": "META"},	
{"name" : "Alphabet Inc. (Class C)","ticker":  "GOOG"},	
{"name" : "Tesla, Inc.", "ticker": "TSLA"},
{"name" : "Walmart", "ticker": "WMT"},
{"name" : "JPMorgan Chase",	"ticker": "JPM"},	
{"name" : "Oracle Corporation",	"ticker": "ORCL"},	
{"name" : "Visa Inc.", "ticker": "V"},
{"name" : "Mastercard",	"ticker": "MA"},
{"name" : "Netflix", "ticker": "NFLX"},	
{"name" : "ExxonMobil",	"ticker": "XOM"},	
{"name" : "Johnson & Johnson", "ticker": "JNJ"},	
{"name" : "Palantir Technologies", "ticker": "PLTR"},	
{"name" : "Costco",	"ticker": "COST"},	
{"name" : "Home Depot",	"ticker": "HD"},	
{"name" : "Bank of America", "ticker": "BAC"},	
{"name" : "Procter & Gamble", "ticker": "PG"},	
{"name" : "Chevron Corporation", "ticker": "CVX"},	
{"name" : "Coca-Cola Company", "ticker": "KO"},	
{"name" : "Cisco", "ticker": "CSCO"},	
{"name" : "Wells Fargo","ticker": "WFC"},
{"name" : "IBM", "ticker": "IBM"},	
{"name" : "T-Mobile US", "ticker": "TMUS"},	
{"name" : "Morgan Stanley",	"ticker": "MS"},	
{"name" : "Salesforce",	"ticker": "CRM"},	
{"name" : "Caterpillar Inc.","ticker": "CAT"},
{"name" : "American Express", "ticker": "AXP"},
{"name" : "Philip Morris International","ticker":  "PM"},	
{"name" : "Goldman Sachs", "ticker": "GS"},	
{"name" : "RTX Corporation", "ticker": "RTX"},	
{"name" : "Abbott Laboratories", "ticker": "ABT"},
{"name" : "McDonald's",	"ticker": "MCD"},	
{"name" : "PepsiCo", "ticker": "PEP"},
{"name" : "Walt Disney Company","ticker":  "DIS"},	
{"name" : "ServiceNow",	"ticker": "NOW"},
{"name" : "AT&T", "ticker": "T"},
{"name" : "Intel", "ticker": "INTC"},
]

In [ ]:
# Calling API for top 40 Companies

stock_price_today = PySimFin()
price_data = []

for company in top_40_companies:
    print(company['ticker']) #Can get rid of this later on
    ticker = str(company['ticker'])
    ind_stock_price = stock_price_today.get_share_prices_today(ticker) #Should use this one --> issue is that on weekends it will be empty
    #ind_stock_price = stock_price_today.get_share_prices_verbose(ticker, '2025-11-07','2025-11-08')
    price_data.extend(ind_stock_price)
    time.sleep(0.5) #Need to pause execution for 0.5 seconds as only 2 requests are alowed per minute on SymFin.

price_data

TRANSFORM

In [ ]:
#This function is definite. it works well

def transformations(price_data: list):
    #creates columns
    dict_data = price_data[0]
    columns = ['name', 'ticker', 'currency']
    info = dict_data['columns']
    columns.extend(info)
    df = pd.DataFrame(columns=columns)
    
    #Appends the data as rows to the DF
    for i in price_data:
        data = [i['name'], i['ticker'], i['currency']]
        stock_data = i['data'][0]
        data.extend(stock_data)
        df.loc[len(df)] = data

    return df #df here is a local variable

In [ ]:
df =transformations(price_data) #here we create the df to be able to save it
df
#Need to fix in case of all null values for a ticker

LOAD

In [ ]:
#Only write to csv for now. Later on we will send to Postgres and PowerBI
df.to_csv('data/output.csv', index=False)

trying out config

In [3]:
import yaml
from pathlib import Path

In [19]:
def load_config(config_path: str = "config.yaml") -> dict:
    """
    Loads config file to centralize and control project behavior.

    Why: Externalizing settings avoids hardcoding parameters across modules,
    supports easier collaboration, and allows quick updates or environment
    switches without touching the core logic.
    """
    if not os.path.isfile(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return config

In [20]:
config = load_config('config.yaml')

In [21]:
api_cfg = config.get('api_information')['headers']
api_cfg

{'accept': 'application/json'}

In [ ]:
#   NEED TO DO HEADERS NOW, WITH API_KEY

In [32]:
#constructing endpoint_url --> use in script.
base_url= config.get('api_information')['base_url']
endpoint = config.get('api_information')['endpoint_url']
endpoint_url = f"{base_url}{endpoint_url}"
endpoint_url

'https://backend.simfin.com/api/v3/companies/prices/compact'

In [35]:
#constructing header   use in script.
headers = config.get('api_information')['headers']
headers['Authorization'] = f"{API_KEY}"


In [33]:
#Creating PySimFin class. This is the API wrapper we will use to make API calls.

class PySimFin:
    def __init__(self):
        self.endpoint_url = f"{endpoint_url}"
        self.headers = headers.copy()   

    def get_share_prices(self, ticker: str):
        params = {'ticker': ticker}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

    def get_share_prices_verbose(self, ticker: str, start: str, end: str):
        params = {'ticker': ticker,
                  'start': start,
                  'end': end}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
    

    def get_share_prices_today(self, ticker: str):
        today = str(datetime.datetime.today()).split()[0] #gets today's date
        params = {'ticker': ticker,
                  'start': today}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

In [ ]:
base_url = "https://backend.simfin.com/api/v3/companies"

endpoint_url= f"{base_url}/prices/compact"

headers = {"accept": "application/json",
            "Authorization": f"{API_KEY}"}

In [26]:
# This will enventually need to go to the config.yaml
top_40_companies =[
{"name" : "Nvidia","ticker": "NVDA"},
{"name" : "Apple Inc.","ticker": "AAPL"},
{"name" : "Microsoft",	"ticker": "MSFT"},
{"name" : "Amazon",	"ticker": "AMZN"},
{"name" : "Meta Platforms",	"ticker": "META"},	
{"name" : "Alphabet Inc. (Class C)","ticker":  "GOOG"},	
{"name" : "Tesla, Inc.", "ticker": "TSLA"},
{"name" : "Walmart", "ticker": "WMT"},
{"name" : "JPMorgan Chase",	"ticker": "JPM"},	
{"name" : "Oracle Corporation",	"ticker": "ORCL"},	
{"name" : "Visa Inc.", "ticker": "V"},
{"name" : "Mastercard",	"ticker": "MA"},
{"name" : "Netflix", "ticker": "NFLX"},	
{"name" : "ExxonMobil",	"ticker": "XOM"},	
{"name" : "Johnson & Johnson", "ticker": "JNJ"},	
{"name" : "Palantir Technologies", "ticker": "PLTR"},	
{"name" : "Costco",	"ticker": "COST"},	
{"name" : "Home Depot",	"ticker": "HD"},	
{"name" : "Bank of America", "ticker": "BAC"},	
{"name" : "Procter & Gamble", "ticker": "PG"},	
{"name" : "Chevron Corporation", "ticker": "CVX"},	
{"name" : "Coca-Cola Company", "ticker": "KO"},	
{"name" : "Cisco", "ticker": "CSCO"},	
{"name" : "Wells Fargo","ticker": "WFC"},
{"name" : "IBM", "ticker": "IBM"},	
{"name" : "T-Mobile US", "ticker": "TMUS"},	
{"name" : "Morgan Stanley",	"ticker": "MS"},	
{"name" : "Salesforce",	"ticker": "CRM"},	
{"name" : "Caterpillar Inc.","ticker": "CAT"},
{"name" : "American Express", "ticker": "AXP"},
{"name" : "Philip Morris International","ticker":  "PM"},	
{"name" : "Goldman Sachs", "ticker": "GS"},	
{"name" : "RTX Corporation", "ticker": "RTX"},	
{"name" : "Abbott Laboratories", "ticker": "ABT"},
{"name" : "McDonald's",	"ticker": "MCD"},	
{"name" : "PepsiCo", "ticker": "PEP"},
{"name" : "Walt Disney Company","ticker":  "DIS"},	
{"name" : "ServiceNow",	"ticker": "NOW"},
{"name" : "AT&T", "ticker": "T"},
{"name" : "Intel", "ticker": "INTC"},
]

In [34]:
# Calling API for top 40 Companies

stock_price_today = PySimFin()
price_data = []

for company in top_40_companies:
    print(company['ticker']) #Can get rid of this later on
    ticker = str(company['ticker'])
    ind_stock_price = stock_price_today.get_share_prices_today(ticker) #Should use this one --> issue is that on weekends it will be empty
    #ind_stock_price = stock_price_today.get_share_prices_verbose(ticker, '2025-11-07','2025-11-08')
    price_data.extend(ind_stock_price)
    time.sleep(0.5) #Need to pause execution for 0.5 seconds as only 2 requests are alowed per minute on SymFin.

price_data

NVDA
AAPL
MSFT
AMZN
META
GOOG
TSLA
WMT
JPM
ORCL
V
MA
NFLX
XOM
JNJ
PLTR
COST
HD
BAC
PG
CVX
KO
CSCO
WFC
IBM
TMUS
MS
CRM
CAT
AXP
PM
GS
RTX
ABT
MCD
PEP
DIS
NOW
T
INTC


[]